In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, accuracy_score, log_loss, mean_squared_error, mean_absolute_percentage_error
import xgboost as xgb
import shap
import optuna

In [ ]:
#x_data = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/x_data.csv')
x_data = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/feature_df_2.csv')
y_data = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/y_data_rms.csv')

In [ ]:
x_df = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/feature_df_2.csv')

df = x_df.merge(y_data[['source_file', 'bottom_rms']], on='source_file').reset_index(drop=True)

df = df.dropna(subset=['bottom_rms'])

X = df.drop(columns=['source_file', 'bottom_rms'])
y = df['bottom_rms']

train_rem_idx = int(len(X) * 0.8) # 80% for training, 20% for validation and testing
val_test_idx = int(len(X) * 0.9) # 10% for validation, 10% for testing (from the remaining 20%)

X_train = X.iloc[:train_rem_idx]
X_val = X.iloc[train_rem_idx:val_test_idx]
X_test = X.iloc[val_test_idx:]

y_train = y.iloc[:train_rem_idx]
y_val = y.iloc[train_rem_idx:val_test_idx]
y_test = y.iloc[val_test_idx:]

In [ ]:
def objective_top(trial):
    params = {
        "n_estimators": 5000,
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.05, log=True),

        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        #"min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
#
        #"gamma": trial.suggest_float("gamma", 0, 10),
        #"reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        #"reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True),

        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "random_state": 42,
    }
    
    model = xgb.XGBRegressor(**params, early_stopping_rounds=50)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred = model.predict(X_val)

    return sklearn.metrics.mean_pinball_loss(y_val, y_pred, alpha=0.85)
study_new = optuna.create_study(direction="minimize")
study_new.optimize(objective_top, n_trials=30)

In [ ]:
opt_params = {'max_depth': 7, 'learning_rate': 0.02216515234619649, 'subsample': 0.7486921438342046, 'colsample_bytree': 0.7579130284008716}
#opt_params = {'objective': 'reg:quantileerror', 'quantile_alpha': 0.85, 'max_depth': 3, 'learning_rate': 0.0020811378368672815, 'subsample': 0.9085628158888369, 'colsample_bytree': 0.7849486435238545}
model_new = xgb.XGBRegressor(**opt_params, n_estimators=5000, early_stopping_rounds=50, random_state=42)
model_new.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_pred_new = model_new.predict(X_test)
mape = mean_absolute_percentage_error(y_test, y_pred_new)
print(f"MAPE with new model: {mape:.4f}")
#print(f'R2 with new model: {r2_score(y_test, y_pred_new):.4f}')

In [ ]:
x_axis = np.arange(len(y_test))
plt.figure(figsize=(10,4))
plt.plot(x_axis[:200], y_test[:200], label="True")
plt.plot(x_axis[:200], y_pred_new[:200], label="Predicted")
plt.title('True vs Predicted <250ft RMS')
plt.xlabel('Sample Index')
plt.ylabel('RMS')
plt.legend()
plt.show()

In [ ]:
shap_values_new = shap.TreeExplainer(model_new).shap_values(X_val)
shap.summary_plot(shap_values_new, X_val, plot_type="dot", max_display=15)

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────

SPIKE_PERCENTILE = 70                # top X% of y_train are considered spikes
BLEND_SHARPNESS             = 10     # higher = harder switch between models
N_TRIALS                    = 30     # optuna trials per model
N_ESTIMATORS                = 5000   # trees per model
EARLY_STOPPING_ROUNDS       = 50
RANDOM_STATE                = 42

# Best params found by Optuna — paste results here to skip re-tuning
BEST_PARAMS_CLASSIFIER = {'max_depth': 4, 'learning_rate': 0.034575879047178036, 'subsample': 0.6897782482740652, 'colsample_bytree': 0.6926504876850375}

BEST_PARAMS_MEAN = {'max_depth': 8, 'learning_rate': 0.01140631788738348, 'subsample': 0.603904781824156, 'colsample_bytree': 0.8510582023021905}

BEST_PARAMS_SPIKE = {'max_depth': 7, 'learning_rate': 0.037336170270069094, 'subsample': 0.6400568787844034, 'colsample_bytree': 0.7015833093192322}

# Set to True to skip Optuna and use BEST_PARAMS above directly
SKIP_OPTUNA = True

In [ ]:

# --- Define spike threshold ---
SPIKE_THRESHOLD = np.percentile(y_train, SPIKE_PERCENTILE)  # tune this

# --- Labels for classifier ---
spike_train = (y_train > SPIKE_THRESHOLD).astype(int)
spike_val   = (y_val   > SPIKE_THRESHOLD).astype(int)
spike_test  = (y_test  > SPIKE_THRESHOLD).astype(int)


# ── 1. SPIKE CLASSIFIER ──────────────────────────────────────────────────────

def objective_classifier(trial):
    params = {
        "n_estimators":     5000,
        "max_depth":        trial.suggest_int("max_depth", 2, 6),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.05, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "objective":        "binary:logistic",
        "eval_metric":      "auc",
        "random_state":     42,
        "scale_pos_weight": (1 - spike_train.mean()) / spike_train.mean(),  # handle class imbalance
    }
    model = xgb.XGBClassifier(**params, early_stopping_rounds=50)
    model.fit(X_train, spike_train, eval_set=[(X_val, spike_val)], verbose=False)
    prob = model.predict_proba(X_val)[:, 1]
    # Reward both precision on spikes and not over-triggering
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(spike_val, prob)

if SKIP_OPTUNA:
    best_params = BEST_PARAMS_CLASSIFIER
else:
    study_clf = optuna.create_study(direction="maximize")
    study_clf.optimize(objective_classifier, n_trials=30)
    best_params = study_clf.best_params

classifier = xgb.XGBClassifier(
    **best_params,
    n_estimators=5000,
    early_stopping_rounds=50,
    scale_pos_weight=(1 - spike_train.mean()) / spike_train.mean(),
)
classifier.fit(X_train, spike_train, eval_set=[(X_val, spike_val)], verbose=False)


# ── 2. MEAN MODEL (normal samples only) ──────────────────────────────────────

mask_train_normal = spike_train == 0
mask_val_normal   = spike_val   == 0

def objective_mean(trial):
    params = {
        "n_estimators":     5000,
        "max_depth":        trial.suggest_int("max_depth", 2, 8),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.05, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "objective":        "reg:squarederror",
        "eval_metric":      "rmse",
        "random_state":     42,
    }
    model = xgb.XGBRegressor(**params, early_stopping_rounds=50)
    model.fit(
        X_train[mask_train_normal], y_train[mask_train_normal],
        eval_set=[(X_val[mask_val_normal], y_val[mask_val_normal])],
        verbose=False
    )
    y_pred = model.predict(X_val[mask_val_normal])
    return r2_score(y_val[mask_val_normal], y_pred)

if SKIP_OPTUNA:
    best_params_mean = BEST_PARAMS_MEAN
else:
    study_mean = optuna.create_study(direction="maximize")
    study_mean.optimize(objective_mean, n_trials=30)
    best_params_mean = study_mean.best_params

model_mean = xgb.XGBRegressor(**best_params_mean, n_estimators=5000, early_stopping_rounds=50)
model_mean.fit(
    X_train[mask_train_normal], y_train[mask_train_normal],
    eval_set=[(X_val[mask_val_normal], y_val[mask_val_normal])],
    verbose=False
)


# ── 3. SPIKE MODEL (spike samples only, weighted) ────────────────────────────

mask_train_spike = spike_train == 1
mask_val_spike   = spike_val   == 1

def objective_spike(trial):
    params = {
        "n_estimators":     5000,
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.05, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "objective":        "reg:squarederror",
        "eval_metric":      "rmse",
        "random_state":     42,
    }
    model = xgb.XGBRegressor(**params, early_stopping_rounds=50)
    # Weight by how extreme the spike is
    spike_weights = y_train[mask_train_spike] / y_train[mask_train_spike].mean()
    model.fit(
        X_train[mask_train_spike], y_train[mask_train_spike],
        sample_weight=spike_weights,
        eval_set=[(X_val[mask_val_spike], y_val[mask_val_spike])],
        verbose=False
    )
    y_pred = model.predict(X_val[mask_val_spike])
    return r2_score(y_val[mask_val_spike], y_pred)

if SKIP_OPTUNA:
    best_params_spike = BEST_PARAMS_SPIKE
else:
    study_spike = optuna.create_study(direction="maximize")
    study_spike.optimize(objective_spike, n_trials=30)
    best_params_spike = study_spike.best_params

model_spike = xgb.XGBRegressor(**best_params_spike, n_estimators=5000, early_stopping_rounds=50)
spike_weights = y_train[mask_train_spike] / y_train[mask_train_spike].mean()
model_spike.fit(
    X_train[mask_train_spike], y_train[mask_train_spike],
    sample_weight=spike_weights,
    eval_set=[(X_val[mask_val_spike], y_val[mask_val_spike])],
    verbose=False
)


# ── 4. BLEND PREDICTIONS ─────────────────────────────────────────────────────

def predict_blend(X, blend_sharpness=10):
    spike_prob  = classifier.predict_proba(X)[:, 1]  # 0–1 probability of being a spike
    pred_mean   = model_mean.predict(X)
    pred_spike  = model_spike.predict(X)

    # Soft blend: weight = sigmoid-sharpened spike probability
    # blend_sharpness controls how hard the switch is (higher = more binary)
    weight = 1 / (1 + np.exp(-blend_sharpness * (spike_prob - 0.5)))

    return (1 - weight) * pred_mean + weight * pred_spike

y_pred_blend = predict_blend(X_test, BLEND_SHARPNESS)

print(f"R²   (blend): {r2_score(y_test, y_pred_blend):.4f}")
print(f"MAPE (blend): {mean_absolute_percentage_error(y_test, y_pred_blend):.4f}")

# Compare spike-specific performance
print(f"\n--- Spike samples only (top 15%) ---")
print(f"R² (blend):  {r2_score(y_test[spike_test==1], y_pred_blend[spike_test==1]):.4f}")

x_axis = np.arange(len(y_test))
plt.figure(figsize=(10,4))
plt.plot(x_axis[:200], y_test[:200], label="True Values")
plt.plot(x_axis[:200], y_pred_blend[:200], label="Blended Predicted")
plt.xlabel('Sample Index')
plt.ylabel('RMS Vertical Acceleration')
plt.legend()
plt.title(f'XGBoost hybrid model ($R^2$ = {r2_score(y_test, y_pred_blend):.4f})')
plt.savefig('xgb_blend_predictions_full.png')
plt.show()

In [ ]:
plt.plot(y_test, y_pred_blend, 'o')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('True Values')
plt.ylabel('Blended Predicted Values')
plt.title('True vs Blended Predicted Values')
plt.savefig('xgb_blend_scatter.png')
plt.legend()
plt.show()


In [ ]:
from Turbulence_categories import evaluate_turbulence_categories
results = evaluate_turbulence_categories(y_test, y_pred_blend)
print(f"Overall Accuracy: {results['overall_accuracy_percent']:.2f}%\n")
print("Confusion Matrix:")
print(results["confusion_matrix"])
print("\nCategory-wise Stats:")
for category, stats in results["category_stats"].items():
    print(f"{category}: Total={stats['total']}, Correct={stats['correct']}, Wrong={stats['wrong']}, Accuracy={stats['accuracy_percent']:.2f}%")